# 17-620 Lab 2 — Encoding Classical Data and Quantum Similarity

**80 minutes | PennyLane | Fall 2026**

### Learning objectives
1. Explain the difference between a classical vector and a quantum feature map.
2. Construct angle and amplitude embeddings with PennyLane.
3. Implement an inverse-circuit overlap kernel and visualize its matrix.
4. Estimate the kernel with finite-shot measurements and discuss practical limitations.

**Textbook:** *A Practical Guide to Quantum Machine Learning and Quantum Optimization*, Chapter 9. **No Qiskit.**

Run the cells in order. The notebook starts with a **fully worked example**. Later sections contain TODOs and checks. Use a new Markdown cell under each conceptual question to record your answer.

> **Key idea:** An encoding is a map `x → |φ(x)⟩`. Different maps produce different similarity measures; a small quantum circuit alone does not establish quantum advantage.

In [ ]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

print("PennyLane", qml.__version__)

## Part 1 — Classical baseline (10 minutes)

Our four classical input vectors are rows of `X`. Cosine similarity measures the *direction* between two vectors: +1 is parallel, 0 is orthogonal, −1 is opposite.

The heatmap has **data-point indices** along both axes. Compare the entries involving vectors 0 and 3.

In [ ]:
X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [-1.0, 0.0]])
cosine = cosine_similarity(X)
print("Classical cosine similarity:\n", np.round(cosine, 3))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cosine, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(X)), labels=[f"x{i}" for i in range(len(X))])
ax.set_yticks(range(len(X)), labels=[f"x{i}" for i in range(len(X))])
ax.set_title("Classical cosine similarity")
fig.colorbar(im, ax=ax, label="cosine similarity")
plt.show()

### Interactive classical cosine-similarity explorer

**Before introducing quantum states, explore classical vector geometry.** Adjust the four Cartesian components of vectors **a** and **b** below. Watch the vectors, their included angle, the dot product, and cosine similarity change together.

$$
\operatorname{cosSim}(a,b)=\frac{a\cdot b}{\lVert a\rVert\,\lVert b\rVert},\qquad
a\cdot b=a_xb_x+a_yb_y.
$$

Try **parallel** vectors, **perpendicular** vectors, and **opposite** vectors. If either vector becomes zero, cosine similarity is **undefined**, not zero. Next, we will compare this signed classical similarity to quantum squared overlap.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def classical_cosine_explorer(a_x=4.0, a_y=1.0, b_x=-1.0, b_y=3.0):
    a = np.array([a_x, a_y], dtype=float)
    b = np.array([b_x, b_y], dtype=float)
    dot = float(np.dot(a, b))
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    valid = norm_a > 1e-12 and norm_b > 1e-12
    cosine = float(np.clip(dot / (norm_a * norm_b), -1, 1)) if valid else None
    theta = float(np.degrees(np.arccos(cosine))) if valid else None

    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    for v, label, color in [(a, "a", "tab:blue"), (b, "b", "tab:green")]:
        ax.annotate("", xy=tuple(v), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=2.8, mutation_scale=18))
        if np.linalg.norm(v) > 1e-12:
            ax.text(v[0] * 1.08 + 0.1, v[1] * 1.08 + 0.1, label,
                    fontsize=14, color=color, weight="bold")
    ax.axhline(0, color="gray", lw=0.9)
    ax.axvline(0, color="gray", lw=0.9)
    ax.set(xlim=(-7, 7), ylim=(-7, 7), aspect="equal",
           xlabel="x component", ylabel="y component",
           title="Two classical feature vectors")
    ax.grid(alpha=0.22)
    plt.show()

    print(f"Dot product: a·b = ({a_x:g})({b_x:g}) + ({a_y:g})({b_y:g}) = {dot:.3f}")
    print(f"Vector magnitudes: ||a|| = {norm_a:.3f}, ||b|| = {norm_b:.3f}")
    if not valid:
        print("Cosine similarity: undefined (at least one vector is zero).")
        return
    if np.isclose(cosine, 1.0, atol=1e-10):
        direction = "same direction"
    elif np.isclose(cosine, -1.0, atol=1e-10):
        direction = "opposite directions"
    elif np.isclose(cosine, 0.0, atol=1e-10):
        direction = "orthogonal"
    elif cosine > 0:
        direction = "acute angle / positive similarity"
    else:
        direction = "obtuse angle / negative similarity"
    print(f"cosSim(a,b) = {cosine:+.3f} | Angle = {theta:.1f}° | {direction}")

component_controls = {
    "a_x": widgets.FloatSlider(value=4, min=-6, max=6, step=0.5, description="aₓ",
                               continuous_update=False),
    "a_y": widgets.FloatSlider(value=1, min=-6, max=6, step=0.5, description="aᵧ",
                               continuous_update=False),
    "b_x": widgets.FloatSlider(value=-1, min=-6, max=6, step=0.5, description="bₓ",
                               continuous_update=False),
    "b_y": widgets.FloatSlider(value=3, min=-6, max=6, step=0.5, description="bᵧ",
                               continuous_update=False),
}
component_output = widgets.interactive_output(classical_cosine_explorer, component_controls)
display(widgets.VBox([widgets.HTML("<b>Drag to change vector components</b>"),
                      *component_controls.values(), component_output]))

### Interactive vector explorer: cosine similarity vs squared overlap

For real, normalized vectors encoded as quantum-state amplitudes, **squared overlap is cosine similarity squared**. Drag the angle to compare the two; at 180°, cosine is −1 while squared overlap is +1 because a global minus sign does not distinguish physical quantum states.

Try 0°, 45°, 90°, 135°, 180°. Note: this correspondence is specific to real amplitude encoding, not arbitrary quantum feature maps.

In [ ]:
import ipywidgets as widgets

def explore_similarity(angle=60):
    theta = np.deg2rad(angle)
    x = np.array([1., 0.])
    y = np.array([np.cos(theta), np.sin(theta)])
    cosine = float(x @ y)
    fidelity = cosine**2
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.arrow(0, 0, x[0], x[1], color="teal", width=0.01, head_width=0.07, length_includes_head=True)
    ax.arrow(0, 0, y[0], y[1], color="darkorange", width=0.01, head_width=0.07, length_includes_head=True)
    ax.set(xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), aspect="equal", xlabel="Feature 1",
           ylabel="Feature 2", title=f"Angle {angle}° | cosine {cosine:+.3f} | squared overlap {fidelity:.3f}")
    ax.grid(alpha=0.25)
    plt.show()

widgets.interact(explore_similarity, angle=widgets.IntSlider(value=60, min=0, max=180, step=5, continuous_update=False));

**Question 1.** Which pairs are orthogonal? What does a negative cosine similarity mean?

## Part 2 — Angle encoding (20 minutes)

### Worked example: one feature → one qubit

A classical scalar `θ` becomes the angle of an `RY(θ)` rotation applied to `|0⟩`.

```text
classical θ = 0          |0⟩ ──RY(0)─────── measure → 0 with probability 1
classical θ = π/2        |0⟩ ──RY(π/2)───── measure → 0 or 1, equally likely
classical θ = π          |0⟩ ──RY(π)─────── measure → 1 with probability 1
```

The resulting state is `cos(θ/2)|0⟩ + sin(θ/2)|1⟩`. Read the output of this worked cell and compare it with the mathematical prediction.

In [ ]:
worked_dev = qml.device("default.qubit", wires=1)

@qml.qnode(worked_dev)
def worked_rotation(theta):
    qml.RY(theta, wires=0)
    return qml.probs(wires=[0])

for theta in [0.0, np.pi / 2, np.pi]:
    print(f"theta={theta:.3f}: P(0),P(1)={worked_rotation(theta)}")
print(qml.draw(worked_rotation)(np.pi / 2))

### Exercise 2A: two features → two qubits

Now represent each input pair with two Y rotations using `qml.AngleEmbedding`. The function should return the four computational-basis probabilities in the order `00, 01, 10, 11`.

**TODO:** Replace `pass` and then run the checks. If a check fails, inspect your operation and wire order.

In [ ]:
angle_dev = qml.device("default.qubit", wires=2)

@qml.qnode(angle_dev)
def angle_state(x):
    # TODO: replace pass with qml.AngleEmbedding(features=x, wires=[0, 1], rotation="Y")
    pass
    return qml.probs(wires=[0, 1])

angle_inputs = np.array([[0.0, 0.0], [np.pi/2, 0.0],
                         [0.0, np.pi/2], [np.pi/2, np.pi/2]])

print("Example:", angle_state(angle_inputs[0]))
print(qml.draw(angle_state)(angle_inputs[1]))
# Self-check: uncomment after implementing
# assert np.allclose(angle_state(angle_inputs[0]), [1, 0, 0, 0])
# assert np.allclose(angle_state(angle_inputs[3]), [0.25, 0.25, 0.25, 0.25])

**Question 2.** Why are features represented as rotation angles? How would you rescale raw input data to radians? What happens to measurement probabilities if you add `2π` to an angle?

## Part 3 — Quantum overlap kernel (20 minutes)

Rather than measuring each encoded state separately, apply the encoding of `x`, undo the encoding of `y`, and measure the probability of the `|00⟩` outcome.

`|00⟩ ── U(x) ── U(y)† ── measure P(00)`

This probability is **`K(x,y)=|⟨φ(y)|φ(x)⟩|²`**. It is a *squared overlap*, not ordinary signed cosine similarity.

**TODO:** Implement the two encoding steps, compute the kernel matrix, draw a heatmap, and uncomment the checks.

### Why is it called **squared overlap**?

A quantum-state inner product, **the overlap amplitude**, is \(s(x,y)=\langle\phi(y)|\phi(x)\rangle\). This amplitude can be negative or complex; it is *not* directly a probability.

The **Born rule** says the probability of projecting the state prepared from x onto the state prepared from y is

$$
\boxed{K_Q(x,y)=|\langle\phi(y)|\phi(x)\rangle|^2.}
$$

*Squared overlap* means **square the absolute value of the inner product**, not square each vector component. If the amplitude is \(0.6+0.3i\), its squared magnitude is \(0.6^2+0.3^2=0.45\). This is the *fidelity* of two pure states.

**Why our circuit works:** Prepare \(U(x)|0\ldots0\rangle=|\phi(x)\rangle\), apply \(U(y)^\dagger\), then measure the probability of all zeros. The amplitude of the all-zero result is

$$
\langle0\ldots0|U(y)^\dagger U(x)|0\ldots0\rangle=\langle\phi(y)|\phi(x)\rangle.
$$

Squaring its magnitude yields exactly the probability we measure.

For **nonzero, real-valued, amplitude-encoded classical vectors**, their quantum overlap amplitude equals the classical cosine similarity. Thus

$$
K_Q(x,y)=\left(\frac{x\cdot y}{\|x\|\|y\|}\right)^2=\operatorname{cosSim}(x,y)^2.
$$

That equivalence **does not apply to every quantum feature map**. In particular, angle encoding generally gives a different function. A vector and its negative give squared overlap 1 under amplitude encoding, because global phase does not distinguish physical quantum states.

**Think:** If the overlap amplitude is \(-0.8\), what probability would the overlap circuit report?

In [ ]:
kernel_dev = qml.device("default.qubit", wires=2)

@qml.qnode(kernel_dev)
def angle_overlap(x, y):
    # TODO: qml.AngleEmbedding for x
    # TODO: qml.adjoint(qml.AngleEmbedding) for y
    return qml.expval(qml.Projector([0, 0], wires=[0, 1]))

def make_kernel(points, circuit):
    return np.array([[circuit(x, y) for y in points] for x in points])

# TODO: uncomment once the function is implemented
# K_angle = make_kernel(angle_inputs, angle_overlap)
# print(np.round(K_angle, 3))
# fig, ax = plt.subplots(figsize=(5, 4))
# im = ax.imshow(K_angle, vmin=0, vmax=1, cmap="viridis")
# ax.set_xticks(range(4), labels=[f"x{i}" for i in range(4)])
# ax.set_yticks(range(4), labels=[f"x{i}" for i in range(4)])
# ax.set_title("Quantum angle-overlap kernel")
# fig.colorbar(im, ax=ax, label="P(00)")
# plt.show()
# assert np.allclose(K_angle, K_angle.T, atol=1e-8)
# assert np.allclose(np.diag(K_angle), np.ones(4), atol=1e-8)
# assert np.all((K_angle >= -1e-8) & (K_angle <= 1 + 1e-8))
# assert np.isclose(angle_overlap(angle_inputs[0], angle_inputs[1]), 0.5)

### The encoding determines the similarity function

For normalized real amplitude encoding the kernel is cos²(θ), where θ is the angle between vectors. For one-qubit Y-angle encoding, using rotation parameters a and b, the kernel is cos²((a−b)/2). These are different similarities! The graph shows the difference as the angle varies.

In [ ]:
delta = np.linspace(0, np.pi, 361)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.degrees(delta), np.cos(delta), label="Cosine similarity")
ax.plot(np.degrees(delta), np.cos(delta)**2, label="Amplitude kernel: cos²(delta)")
ax.plot(np.degrees(delta), np.cos(delta/2)**2, label="RY angle kernel: cos²(delta/2)")
ax.set(xlabel="Angle / rotation difference (degrees)", ylabel="Similarity / squared overlap")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

**Question 3.** Is squared overlap the same as Euclidean distance or classical cosine similarity? Which information disappears when we square an inner product?

## Part 4 — Quantum kernel using amplitude encoding (15 minutes)

**This repeats the full kernel construction from Part 3, now using amplitude encoding.** The only change is the feature map.

1. Load vector a with AmplitudeEmbedding, then apply the inverse embedding for vector b.
2. Measure the projector onto |00⟩ to obtain K_amplitude(a,b), the squared state overlap.
3. Build the **amplitude quantum kernel matrix** for every pair of vectors: K_amplitude = make_kernel(A, amplitude_overlap).
4. Compare it against the squared *classical* cosine-similarity matrix using side-by-side heatmaps with identical color scales.

For nonzero real vectors, K_amplitude(a,b) = cosSim(a,b)². Here four amplitudes fit on two qubits; state preparation can still be expensive.

**TODO:** Complete the two operations, calculate both matrices, and compare them visually.

In [ ]:
amp_dev = qml.device("default.qubit", wires=2)

@qml.qnode(amp_dev)
def amplitude_overlap(a, b):
    # TODO: qml.AmplitudeEmbedding(a, wires=[0, 1], normalize=True)
    # TODO: qml.adjoint(qml.AmplitudeEmbedding)(b, wires=[0, 1], normalize=True)
    return qml.expval(qml.Projector([0, 0], wires=[0, 1]))

A = np.array([[1., 0., 0., 0.], [0., 1., 0., 0.],
              [1., 1., 0., 0.], [-1., 0., 0., 0.]])

# TODO: calculate the quantum AMPLITUDE kernel matrix (as in Part 3).
# K_amplitude = make_kernel(A, amplitude_overlap)
# K_classical_squared = cosine_similarity(A)**2

# TODO: uncomment and plot the matrices on an identical [0,1] color scale
# fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# for ax, matrix, title in zip(axes, [K_amplitude, K_classical_squared],
#                              ["Quantum amplitude kernel", "Classical cosine²"]):
#     im = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
#     ax.set_title(title)
#     ax.set_xticks(range(4), labels=[f"a{i}" for i in range(4)])
#     ax.set_yticks(range(4), labels=[f"a{i}" for i in range(4)])
# fig.colorbar(im, ax=axes)
# plt.show()

# assert np.allclose(K_amplitude, K_classical_squared, atol=1e-7)
# assert np.isclose(K_amplitude[0, 3], 1.0)

### What is optimized? The quantum kernel is **fixed** in Lab 2

Our angle and amplitude feature maps have **no trainable parameters**. The data determine the circuit operations, and the squared overlap gives a **fixed quantum kernel matrix** for those data points (finite shots add sampling noise). **Evaluating this matrix is not itself training a model.**

To classify labeled examples, a **quantum-kernel SVM (QSVM)** can pass that matrix to the *same classical SVM optimizer* used with a classical kernel. A binary soft-margin SVM learns coefficients \(\alpha_i\), with labels \(y_i\in\{-1,+1\}\):

$$
\max_{\alpha}\;\sum_i\alpha_i-\frac12\sum_{i,j}\alpha_i\alpha_jy_iy_jK(x_i,x_j),
\qquad 0\le\alpha_i\le C,\quad \sum_i\alpha_i y_i=0.
$$

Here the **kernel values remain fixed**, while the **SVM coefficients are optimized** to produce a separating decision boundary. The regularization setting \(C\) is a hyperparameter; we do not train any quantum gates in this lab.

**Could a quantum kernel be trainable?** Yes. Another approach introduces circuit parameters \(\theta\) into \(U(x;\theta)\) and uses an outer training objective to tune them, recomputing the kernel as the representation changes. A variational quantum classifier is a different architecture that learns circuit parameters to predict labels directly. **Neither is implemented here.**

**No quantum advantage is implied:** Our real amplitude kernel is just classical cosine similarity squared, which can be computed efficiently without a quantum computer.

**Discussion:** If you trained an SVM using this lab's kernel matrix, what would be learned: quantum rotation gates, pairwise similarities, or the classical SVM coefficients?

**Question 4.** Why do a vector and its negative give the same squared overlap? How does state-preparation cost affect the overall algorithm?

## Part 5 — Sampling the overlap (15 minutes)

The overlap circuit returns an exact `P(00)` on an ideal analytic simulator. Real measurements instead produce bitstrings. Estimate `P(00)` by counting `00` outcomes and dividing by the shot count.

Use `qml.set_shots` on the QNode to avoid deprecated device-shot settings in PennyLane 0.45.1. <https://docs.pennylane.ai/en/stable/code/api/pennylane.workflow.set_shots.html>

**TODO:** Implement the same inverse-encoding steps, then compute the empirical frequency for 1000 shots.

In [ ]:
shots = 1000
sample_dev = qml.device("default.qubit", wires=2)

@qml.set_shots(shots=shots)
@qml.qnode(sample_dev)
def sampled_angle_overlap(x, y):
    # TODO: embed x and undo the embedding of y
    return qml.sample(wires=[0, 1])

x, y = angle_inputs[0], angle_inputs[1]
# TODO: samples = sampled_angle_overlap(x, y)
# TODO: estimate = np.mean(np.all(samples == 0, axis=1))
# TODO: compare to angle_overlap(x, y); expect sampling variation
# assert samples.shape == (shots, 2)
# assert abs(estimate - angle_overlap(x, y)) < 0.12

## Exit ticket (submit as Markdown answers)

1. Exactly what does the quantum overlap kernel measure?
2. Why doesn't logarithmic qubit count for amplitude encoding automatically imply a quantum speedup?
3. How does finite-shot sampling affect the estimated kernel? What happens if we increase shots?
4. Give one fair classical baseline for a future learning task.

**Optional extension:** Train a classifier with a precomputed quantum kernel and compare to a classical kernel on the same train/test split. Do not equate accurate classification with quantum advantage.

**Submission:** Save and submit your completed notebook as instructed in the course LMS.